# spi_nb_silver_energy

**Purpose.** Silver transformation of the REE demand series for the Energy
source (source 3). Filters the demand series, derives year and month, casts
the value, maps regions to the `spi_dim_region` vocabulary (ADR-010 §4), and
publishes complete months only (ADR-015).

**Inputs.** Delta table `spi_bronze_energy_raw` in the default Lakehouse, all
columns string.

**Outputs.** Delta table `spi_silver_energy_cleaned` in the default Lakehouse,
written with `mode="overwrite"`. Columns: `region_name`, `indicator_category`,
`year`, `month`, `indicator_value`, `_ingestion_timestamp`.

**Dependencies.** `pandas`, `deltalake`. Runs on the Fabric **Python** runtime
(no Spark session), per ADR-012 (2026-10-04 addendum).

**Exit contract.** On success, calls `notebookutils.notebook.exit()` with
`{"rows_written": <int>, "dropped_null_zero": <int>, "dropped_partial_month": <int>}`.
On failure, the exception propagates uncaught, per the 2026-09-30 addendum to
ADR-009.

**Last updated.** 2026-10-04.

In [ ]:
import json
import logging

import pandas as pd
from deltalake import DeltaTable, write_deltalake

# ────────────────────────────────────────────────────────────────────────────
# Tables — OneLake abfss:// URL, not /lakehouse/default/ (lessons-learned.md,
# Python notebooks)
# ────────────────────────────────────────────────────────────────────────────

TABLES_URI = (
    "abfss://bbffcd8e-0f61-436d-bee8-eac2d26d38b2@onelake.dfs.fabric.microsoft.com/"
    "c6048c3c-087e-4436-80fa-6013d1a04aeb/Tables"
)
SOURCE_TABLE = f"{TABLES_URI}/spi_bronze_energy_raw"
TARGET_TABLE = f"{TABLES_URI}/spi_silver_energy_cleaned"

# ────────────────────────────────────────────────────────────────────────────
# Transformation constants
# ────────────────────────────────────────────────────────────────────────────

SERIES_TYPE = "Demand"  # Bronze calls the /en/ endpoint
INDICATOR_CATEGORY = "Demanda eléctrica"

# Bronze geo_name -> spi_dim_region.region_name. Values copied from
# spi_df_gold_dimensions, not retyped (ADR-010 §4). Energy 'Nacional' is the
# peninsular system (ADR-015, 2026-10-04 addendum).
REGION_MAP = {
    "Espana": "Nacional",
    "Catalunya": "Cataluña",
    "Madrid": "Madrid",
}

# Bronze stores missing values as strings (df.astype(str) in spi_nb_bronze_energy)
NULL_TOKENS = ["None", "nan", ""]

OUTPUT_COLUMNS = [
    "region_name",
    "indicator_category",
    "year",
    "month",
    "indicator_value",
    "_ingestion_timestamp",
]

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Configure Logging
# ────────────────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger("silver_energy")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create function storage_options and read_bronze
# ────────────────────────────────────────────────────────────────────────────

def storage_options() -> dict:
    """
    Build the OneLake storage options for deltalake.
    Returns:
        dict: Bearer token and Fabric endpoint flag.
    """
    return {
        "bearer_token": notebookutils.credentials.getToken("storage"),
        "use_fabric_endpoint": "true",
    }


def read_bronze() -> pd.DataFrame:
    """
    Read spi_bronze_energy_raw into pandas.
    Returns:
        pd.DataFrame: The Bronze table, all columns string.
    """
    df = DeltaTable(SOURCE_TABLE, storage_options=storage_options()).to_pandas()
    log.info(f"Read spi_bronze_energy_raw: {len(df)} rows")
    return df

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create function transform
# ────────────────────────────────────────────────────────────────────────────

def transform(bronze: pd.DataFrame) -> tuple[pd.DataFrame, int, int]:
    """
    Apply the Silver rules to the Bronze rows.
    Args:
        bronze (pd.DataFrame): The Bronze table.
    Returns:
        tuple: The Silver DataFrame, rows dropped as null or zero, and rows
        dropped as partial months.
    Raises:
        ValueError: On an unmapped geo_name, a non-numeric value, or a
        duplicate (region_name, year, month).
    """
    # 1. Demand series only
    df = bronze[bronze["series_type"] == SERIES_TYPE].copy()
    log.info(f"Series '{SERIES_TYPE}': {len(df)} rows")

    # 2. Year and month from the datetime string. Not via UTC: REE timestamps
    #    carry +01:00/+02:00, and a UTC conversion moves the 1st of the month
    #    into the previous month.
    df["period"] = df["datetime"].str[:7]
    df["year"] = df["period"].str[:4].astype("int64")
    df["month"] = df["period"].str[5:7].astype("int64")

    # 3. Cast the value. Null tokens become NaN; any other non-numeric string raises.
    value = df["value"].where(~df["value"].isin(NULL_TOKENS))
    df["indicator_value"] = pd.to_numeric(value, errors="raise").astype("float64")

    # 4. Region vocabulary. An unmapped geo_name raises.
    unmapped = sorted(set(df["geo_name"]) - set(REGION_MAP))
    if unmapped:
        raise ValueError(f"Unmapped geo_name values: {unmapped}")
    df["region_name"] = df["geo_name"].map(REGION_MAP)

    # 5. Indicator category
    df["indicator_category"] = INDICATOR_CATEGORY

    # 6. Drop null or zero values (Phase 4 DQ-08)
    null_zero = df["indicator_value"].isna() | (df["indicator_value"] == 0)
    dropped_null_zero = int(null_zero.sum())
    df = df[~null_zero]
    log.info(f"Dropped null or zero values: {dropped_null_zero}")

    # 7. Complete months only (ADR-015 Decision 1). The API returns the current
    #    month as a running total. _ingestion_timestamp is UTC, so a run in the
    #    first hours of the 1st (Madrid time) also drops the last complete month;
    #    the next run publishes it.
    ingestion_month = df["_ingestion_timestamp"].str[:7]
    partial = df["period"] >= ingestion_month
    dropped_partial_month = int(partial.sum())
    df = df[~partial]
    log.info(f"Dropped partial-month rows: {dropped_partial_month}")

    # 8. (region_name, year, month) must be unique. Bronze is overwritten each
    #    run, so a duplicate is a defect: raise, do not deduplicate.
    keys = ["region_name", "year", "month"]
    duplicated = df.duplicated(keys, keep=False)
    if duplicated.any():
        sample = df.loc[duplicated, keys].drop_duplicates().head(10).to_dict("records")
        raise ValueError(f"Duplicate (region_name, year, month): {sample}")

    # 9. Madrid values are kept as received (ADR-015 Decision 3). No code.

    df["_ingestion_timestamp"] = (
        pd.to_datetime(df["_ingestion_timestamp"], utc=True).astype("datetime64[us, UTC]")
    )

    return df[OUTPUT_COLUMNS].reset_index(drop=True), dropped_null_zero, dropped_partial_month

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create main to run the entire process
# ────────────────────────────────────────────────────────────────────────────

def main() -> dict:
    """
    Read Bronze, apply the Silver rules, and overwrite spi_silver_energy_cleaned.
    Returns:
        dict: rows_written, dropped_null_zero, dropped_partial_month.
    """
    bronze = read_bronze()
    silver, dropped_null_zero, dropped_partial_month = transform(bronze)
    write_deltalake(TARGET_TABLE, silver, mode="overwrite", storage_options=storage_options())
    log.info(f"Data stored in spi_silver_energy_cleaned: {len(silver)} rows")
    return {
        "rows_written": len(silver),
        "dropped_null_zero": dropped_null_zero,
        "dropped_partial_month": dropped_partial_month,
    }

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Exit contract — outside any try/except (ADR-009, 2026-09-30 addendum)
# ────────────────────────────────────────────────────────────────────────────
# Interactive runs: Fabric marks this cell with ✗ because exit() stops the
# notebook on purpose. Pipeline runs: the activity reports Succeeded and
# passes exitValue to the logging Script activity (ADR-009).

result = main()
notebookutils.notebook.exit(json.dumps(result))